In [ ]:
# LOCAL E2E bootstrap (papermill / nbclient)
import os
from pathlib import Path

# mock = harness+contratos; hf = Construtor real (precisa ~3GB livres)
LLM_BACKEND = os.environ.get("LLM_BACKEND", "mock")
os.environ["LLM_BACKEND"] = LLM_BACKEND

# Garantir cwd = multi-agents-lab (onde fica dutos-do-q/)
here = Path.cwd()
if not (here / "dutos-do-q").exists() and (here / "multi-agents-lab" / "dutos-do-q").exists():
    os.chdir(here / "multi-agents-lab")
print("cwd=", Path.cwd())
print("LLM_BACKEND=", os.environ["LLM_BACKEND"])
assert (Path.cwd() / "dutos-do-q").exists(), "rode a partir de multi-agents-lab/"


# Notebook 1 · Bloco 1 — O Duto Batch e o Agente Construtor

**120 minutos: 60 de conceito, 60 de prática.**

Vocês não vão escrever o pipeline. Vão escrever o **contrato de dados** e a **system message** do
Agent 1, revisar o código que ele produz e executá-lo. O que vale ponto é a decisão, não a digitação.

**Papéis no esquadrão (rotativos, 4 pessoas):**

- **Arquiteto(a)** decide o contrato e a system message. Não escreve código.
- **Builders (2)** operam os notebooks e revisam o que o Construtor gerou.
- **Red Team** lê a quarentena e procura o que passou e não deveria.

**Metas do harness:** bronze 60 · prata 80 · ouro 95. O Caos do professor vale de -20 a +20.

## Passo 1 — preparar a sessão

In [ ]:
# deps: usa venv local; pip so se faltar algo
import importlib, subprocess, sys
_need = []
for _m in ["deltalake", "duckdb", "yaml", "pyarrow", "pandas", "sentence_transformers"]:
    try:
        importlib.import_module(_m if _m != "yaml" else "yaml")
    except ImportError:
        _need.append("pyyaml" if _m == "yaml" else ("sentence-transformers" if _m == "sentence_transformers" else _m))
if _need:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *_need])
    print("installed", _need)
else:
    print("deps ok")

# Passo 1 de 9 — preparar a sessão (roda uma vez, ~90 s)

import os, sys, json, shutil
from pathlib import Path

# O kit vem de um zip. Troque KIT_URL pelo endereço que o professor passar,
# ou faça upload de dutos-do-q.zip no painel de arquivos do Colab (ícone de pasta à esquerda).
KIT_URL = os.environ.get("KIT_URL", "")
RAIZ = Path("/content") if Path("/content").exists() else Path.cwd()
KIT = RAIZ / "dutos-do-q"

if not KIT.exists():
    zip_local = RAIZ / "dutos-do-q.zip"
    if KIT_URL and not zip_local.exists():
        !wget -q -O {zip_local} {KIT_URL}
    assert zip_local.exists(), "Faça upload de dutos-do-q.zip no painel de arquivos, ou preencha KIT_URL."
    shutil.unpack_archive(str(zip_local), str(RAIZ))

sys.path.insert(0, str(KIT / "kit"))
os.chdir(KIT)
print("kit em", KIT)
print(sorted(p.name for p in (KIT / "kit").iterdir()))

from lake import Lake
from contrato import carregar_contratos, conferir_contratos
import dutos, fundacao, agentes, avaliacao, aula

ESQUADRAO = "esquadrao_00"   # <<< TROQUE pelo nome ou número do seu esquadrão

lake = Lake(str(KIT / "lakehouse"))
lake.criar_todas()
INBOX = dutos.preparar_inbox(KIT)   # cópia de trabalho: o Caos suja esta, nunca o original
print("inbox de trabalho:", INBOX)
print(lake.resumo().to_string(index=False))

## Passo 1b — contratos do Drive FONTE (reprodutível)

Baixa os **4 YAML canônicos** da pasta FONTE (compartilhada com link). Qualquer um
roda e cai no mesmo contrato — sem colar arquivo na mão.

Ordem: pasta `contratos/` local (papermill) → **gdown Drive** → GitHub raw (fallback).


In [ ]:
# Passo 1b — contratos FONTE via gdown (link público) + fallbacks
from __future__ import annotations

import os
import subprocess
import sys
import urllib.request
from pathlib import Path

# FONTE: https://drive.google.com/drive/folders/1tHhppoiBjij7vebmbqZpfH8NC68MzQa8
# anyoneWithLink = reader → gdown sem auth
CONTRATOS_DRIVE = {
    "clientes.yaml":    "1gY4mCJGDU8DkNTQ_TM6JNAGpx6FRZyyZ",
    "documentos.yaml":  "19akHATUI711piOvtWMuuIfnGEtdGzBq0",
    "transacoes.yaml":  "1ayDnn7Ycwfsu67cHrKlDP7bjxQndQHGi",
    "tarifas.yaml":     "1Ur58dlXkU9FoTpAFdX_qWt7x4AMKuYL8",
}
CONTRATOS_GIT_REF = os.environ.get("CONTRATOS_GIT_REF", "feat/multi-agents-lab-bloco1")
CONTRATOS_GIT_BASE = (
    "https://raw.githubusercontent.com/gabriel-dantas98/fiap-mba-multi-agent/"
    f"{CONTRATOS_GIT_REF}/multi-agents-lab/contratos"
)

DEST = Path(KIT) / "contratos"
DEST.mkdir(parents=True, exist_ok=True)

_LOCAL = [
    Path.cwd() / "contratos",
    Path.cwd().parent / "contratos",
    Path("/content/multi-agents-lab/contratos"),
]


def _local_ok() -> Path | None:
    for p in _LOCAL:
        if all((p / n).is_file() for n in CONTRATOS_DRIVE):
            return p
    return None


def _ensure_gdown():
    try:
        import gdown  # noqa: F401
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "-q", "install", "gdown"])


def _via_gdown(fid: str, dest: Path) -> None:
    import gdown
    url = f"https://drive.google.com/uc?id={fid}"
    out = gdown.download(url, str(dest), quiet=True, fuzzy=True)
    if not out or not dest.is_file() or dest.stat().st_size < 50:
        raise RuntimeError(f"gdown falhou para {fid}")


def _via_github(nome: str, dest: Path) -> None:
    with urllib.request.urlopen(f"{CONTRATOS_GIT_BASE}/{nome}", timeout=30) as r:
        dest.write_bytes(r.read())


force = os.environ.get("CONTRATOS_SOURCE", "").strip().lower()  # local|drive|github
local = _local_ok()
origens: dict[str, str] = {}

if force == "local" or (not force and local is not None):
    if local is None:
        raise RuntimeError("CONTRATOS_SOURCE=local mas contratos/ incompleto")
    for nome in CONTRATOS_DRIVE:
        dest = DEST / nome
        dest.write_bytes((local / nome).read_bytes())
        origens[nome] = f"local:{local}"
        print(f"  {nome:18s} ← {origens[nome]}  ({dest.stat().st_size} B)")
else:
    if force in ("", "drive"):
        _ensure_gdown()
    for nome, fid in CONTRATOS_DRIVE.items():
        dest = DEST / nome
        ok = False
        if force in ("", "drive"):
            try:
                _via_gdown(fid, dest)
                origens[nome] = "drive:gdown"
                ok = True
            except Exception as e_drive:
                err_drive = e_drive
            else:
                err_drive = None
        else:
            err_drive = None
        if not ok and force in ("", "github", "drive"):
            try:
                _via_github(nome, dest)
                origens[nome] = "github"
                ok = True
            except Exception as e_git:
                err_git = e_git
            else:
                err_git = None
        else:
            err_git = None
        if not ok and local is not None:
            dest.write_bytes((local / nome).read_bytes())
            origens[nome] = f"local:{local}"
            ok = True
        if not ok:
            raise RuntimeError(
                f"falhou {nome}: drive={err_drive!r}; github={err_git!r}"
            )
        print(f"  {nome:18s} ← {origens[nome]}  ({dest.stat().st_size} B)")

print("contratos em", DEST)
print(conferir_contratos())


### A missão, por escrito

In [ ]:
aula.briefing(KIT, "Missão 1")

## Passo 2 — Bronze: o arquivo bruto vira uma linha

A Bronze não interpreta nada. Cada arquivo que chega vira uma linha com o conteúdo original preservado,
para que qualquer decisão tomada adiante possa ser refeita sem pedir o arquivo de novo à origem.

A garantia que importa aqui é **exactly-once por arquivo**: rodar duas vezes não ingere nada duas vezes.
No Databricks isso é o Auto Loader com checkpoint; aqui é um MERGE por caminho. O mecanismo muda, a
garantia não.

In [ ]:
print(dutos.bronze(lake, INBOX))
print(lake.sql("SELECT nome, tamanho FROM bronze.arquivos ORDER BY nome LIMIT 8").to_string(index=False))

In [ ]:
# rode de novo: zero arquivos novos. Se este número não for zero, o duto não é idempotente.
print(dutos.bronze(lake, INBOX))

## Passo 3 — DECISÃO DO ARQUITETO · preencher o contrato

Aqui começa a pontuação. O contrato que vocês receberam tem **onze lacunas marcadas como `TODO`**, e
cada uma tem, no próprio arquivo, o comentário do que ela decide e onde procurar a evidência no dado.

O duto roda com o contrato incompleto. Não dá erro, não avisa, e produz uma Silver de aparência normal.
Ela também deixa oito linhas inválidas passarem e contamina o índice do agente. Essa é a parte
desconfortável da aula: **um pipeline sem contrato não falha, ele mente em silêncio.**

Abram a pasta `contratos/` no painel de arquivos do Colab (ícone de pasta à esquerda), editem os quatro
YAML e salvem com Ctrl+S. Depois rodem a célula de conferência de novo.

In [ ]:
conferir_contratos()

In [ ]:
# Leia um contrato inteiro aqui, se preferir não abrir o arquivo. Troque o nome para ver os outros.
print((KIT / "contratos" / "transacoes.yaml").read_text())

Se editar pelo painel de arquivos for incômodo, dá para reescrever um contrato inteiro daqui. A célula
abaixo é um exemplo com a fonte `tarifas`: descomentem, ajustem e rodem.

In [ ]:
# exemplo de edição pelo notebook (descomente e adapte)
# (KIT / "contratos" / "tarifas.yaml").write_text("""fonte: tarifas
# padrao_arquivo: "tarifas*.parquet"
# formato: parquet
# chave: [tarifa_id]
# schema_drift: quarentena
# duplicatas: manter_primeira
# colunas:
#   tarifa_id:       {tipo: string, obrigatorio: true}
#   tipo:            {tipo: string, obrigatorio: true, dominio: [saque, ted, manutencao, pix]}
#   valor:           {tipo: double, obrigatorio: true, minimo: 0}
#   vigencia_inicio: {tipo: date, obrigatorio: true, formatos: ["%Y-%m-%d"]}
#   vigencia_fim:    {tipo: date, formatos: ["%Y-%m-%d"]}
# regras_conjunto:
#   - {nome: sem_sobreposicao_vigencia, particao: ???, inicio: ???, fim: ???}
# """)

In [ ]:
import yaml
contratos = carregar_contratos()
contrato_yaml = yaml.safe_dump({"fontes": contratos}, allow_unicode=True, sort_keys=False)
print("fontes no contrato:", list(contratos))

## Passo 4 — DECISÃO DO ARQUITETO · a system message do Construtor

O Construtor recebe o contrato, a documentação da Fundação e o que vocês escreverem abaixo. Ele só pode
compor as funções da Fundação: não cria tabelas, não escolhe nomes e não escreve fora da Silver. Essa
coleira é o que torna o resultado avaliável.

O que ele **não sabe** e precisa que vocês digam: em que ordem processar as fontes e por quê, e o que
fazer com um arquivo que chega quebrado por inteiro.

In [ ]:
system_message = """
# INSTRUÇÕES DO ESQUADRÃO · Agent 1 Construtor
# Audiência: Qwen2.5-Coder-1.5B preenchendo TRÊS lacunas do esqueleto.
# Responda só com o código de cada lacuna. Não invente funções, não altere o esqueleto.

## Objetivo
Produzir um pipeline Silver que (1) processa fontes na ordem certa por causa da FK,
(2) passa refs só em transacoes, (3) manda arquivo quebrado inteiro para quarentena.

## Lacuna ___ORDEM_DAS_FONTES___
DECISÃO: clientes antes de transacoes. Sem isso, a FK rejeita quase todas as linhas.

PREENCHA EXATAMENTE com (números 0..3, todos os quatro nomes):
sorted(contratos, key=lambda f: {"clientes": 0, "tarifas": 1, "transacoes": 2, "documentos": 3}.get(f, 9))

PROIBIDO:
- ordem alfabética ou sorted(contratos) sem key
- omitir qualquer das quatro fontes no dict
- transacoes com número menor que clientes
- lista literal ["clientes", ...] (use o sorted acima)

POR QUE tarifas=1 e documentos=3: tarifas não tem FK; documentos fecha o lote antes de finalizar_documentos.

## Lacuna ___REFERENCIAS_PARA_FK___
DECISÃO: refs é um dict. Só transacoes recebe clientes. Demais fontes recebem {}.

PREENCHA EXATAMENTE com:
{"clientes": (ref if ref is not None else fundacao.referencia_clientes(lake))} if fonte == "transacoes" else {}

PROIBIDO:
- refs = ... (não use atribuição; a lacuna JÁ é o lado direito de `refs =`)
- passar refs para todas as fontes
- fundacao.referencia_clientes(lake) solto sem dict
- chave errada (tem que ser "clientes", igual ao fk do contrato)

## Lacuna ___O_QUE_FAZER_COM_O_ARQUIVO_INVALIDO___
DECISÃO: falhou a leitura = arquivo inteiro em quarentena. Nunca engula a exceção.

CONTEXTO DO except: ok, q e df NÃO EXISTEM. Use só e, r, fonte, lake, m.

PREENCHA EXATAMENTE com estas quatro linhas:
fundacao.gravar_quarentena(lake, fonte, r["nome"], None, str(e))
fundacao.marcar_processado(lake, r["path"], fonte, "quarentena", 0, 1)
m["drift"] += 1
m["rows_rejected"] += 1

PROIBIDO:
- len(ok), len(q), len(df) (NameError)
- raise / return / pass / continue (some o arquivo do relatório)
- marcar_processado com status "ok"
- gravar_quarentena sem str(e)

NOTA: o segundo zero do marcar_processado NÃO é zero: use 0 ok e 1 rejeitado (o arquivo conta como uma rejeição).

## Checklist mental (antes de responder cada lacuna)
1. A expressão/bloco cola no esqueleto sem `=` extra na frente?
2. Todas as quatro fontes aparecem na ordem?
3. refs vira dict {"clientes": set} só quando fonte == "transacoes"?
4. No except, zero menção a ok/q/df?
"""
print(system_message)


## Passo 5 — carregar o modelo do Construtor

Qwen2.5-Coder-1.5B rodando dentro do notebook, em CPU. Baixa uma vez por sessão (~3 GB) e depois
responde em segundos por lacuna. Não depende de conta, de token nem de cota.

Enquanto baixa, leiam o esqueleto na célula seguinte: é o que vai ser preenchido.

In [ ]:
# carrega mock ou HF conforme LLM_BACKEND
print(agentes.carregar_modelo("Qwen/Qwen2.5-Coder-1.5B-Instruct"))


In [ ]:
print(agentes.ESQUELETO)

Um modelo de 1,5 bilhão de parâmetros não escreve um módulo inteiro que funcione. Escreve três decisões
específicas, se você perguntar uma de cada vez. O esqueleto fixo é o guardrail mais barato que existe:
troca "escreva o pipeline" por "complete esta lacuna", e o espaço de erro encolhe junto.

Isso não é limitação do exercício. É como se constrói agente de código em produção: contexto estreito,
formato fixo, verificação depois.

## Passo 6 — o Construtor escreve, e o kit testa antes de vocês confiarem

`construir` faz quatro coisas em sequência: gera as três lacunas, passa os guardrails estáticos, roda o
código num lakehouse descartável e compara o resultado com o esperado. Se reprovar, ele gera de novo
**com o diagnóstico na entrada**, até duas vezes.

Por que o teste de fumaça existe: guardrail estático não pega erro de lógica. Um código que processa
`transacoes` antes de `clientes` compila, executa, não levanta exceção nenhuma, e rejeita 2.017 linhas
das 2.243 porque toda transação virou órfã. Sem o teste, isso só aparece no harness, no fim da prática.

Cada tentativa leva de 45 a 90 segundos em CPU. Leiam o esqueleto enquanto roda.

In [ ]:
r = agentes.construir(contrato_yaml, system_message, contratos, KIT, tentativas=2)
print("\nresultado:", "passou" if r["ok"] else "não passou", "· tentativas:", r["tentativas"])
g = {"codigo": r["codigo"]}

In [ ]:
print(r["codigo"])

## Passo 7 — Builders revisam antes de executar

Leiam o código. Três perguntas antes de apertar o botão:

1. A ordem das fontes está certa? Se `transacoes` vier antes de `clientes`, todas as transações viram órfãs.
2. As referências da FK estão sendo passadas só para `transacoes`?
3. Um arquivo quebrado vai inteiro para a quarentena, ou o erro engole o arquivo em silêncio?

O teste de fumaça já respondeu essas perguntas com números. A revisão de vocês é sobre o que fazer a
seguir: se o Construtor errou, **o que faltava na system message?** É essa a pergunta da ficha.

Se o esquadrão travar e o tempo apertar, a última célula adota a referência: vocês perdem os pontos da
geração, não a missão inteira.

In [ ]:
if not r["ok"]:
    print("O Construtor não chegou lá em duas tentativas. O que ele errou:")
    for h in r["historico"]:
        print(" ", h.get("problemas") or h["diagnostico"].get("sintomas") or h["diagnostico"].get("erro"))
    print("\nAjustem a system message do Passo 4 e rodem o Passo 6 de novo, ou usem o plano B abaixo.")
else:
    print("Silver:", agentes.executar(r["codigo"], lake, {"fontes": contratos}))

In [ ]:
# Plano B se o Construtor falhar (perde pontos de geracao, nao a missao):
if "g" not in globals() or not g.get("codigo"):
    g = {"codigo": agentes.codigo_de_referencia()}
elif "r" in globals() and not r.get("ok", True):
    g["codigo"] = agentes.codigo_de_referencia()
print("Silver:", agentes.executar(g["codigo"], lake, {"fontes": contratos}))


## Passo 8 — Red Team: leia a quarentena

A quarentena é o produto mais importante do duto. Uma linha rejeitada sem motivo legível é um chamado
de suporte na semana que vem.

In [ ]:
print(lake.sql("""SELECT fonte, COUNT(*) linhas FROM silver.quarentena GROUP BY 1 ORDER BY 1""").to_string(index=False))
print()
print(lake.sql("""SELECT chave, motivo FROM silver.quarentena ORDER BY chave""").to_string(index=False))

## Passo 9 — Gold: chunks e embeddings, só do que mudou

A Gold é o que o agente lê. Cada documento vira chunks, cada chunk vira um vetor, e cada chunk carrega
`vigente` e `autoritativo`.

O número a observar é `embeds_executados`. Na primeira execução ele é o total. Na segunda precisa ser
**zero**, porque nada mudou. Um duto que re-embeda tudo a cada execução funciona igual e custa dez vezes
mais, e é exatamente o tipo de decisão que ninguém revisa depois que entra em produção.

In [ ]:
print("1ª execução:", dutos.gold(lake))
print("2ª execução:", dutos.gold(lake))

In [ ]:
# a vigência em ação: a v1 da tabela de tarifas continua existindo, mas fora do índice do agente
print(lake.sql("""SELECT doc_id, versao, tipo, vigente, autoritativo FROM silver.documentos
                  WHERE doc_id IN ('prod-tabela-tarifas','mkt-blog-cdb','faq-antigo-tarifas-2023')
                  ORDER BY doc_id, versao""").to_string(index=False))

In [ ]:
# e o efeito disso na busca: a pergunta sobre tarifa cai na versão certa
print(dutos.buscar(lake, "Qual a tarifa de saque em caixa eletrônico?", k=3)[["doc_id","versao","score"]].to_string(index=False))

## Passo 10 — o Caos do professor (aos 40 minutos de prática)

Seis arquivos novos caem no inbox sem aviso: um CSV com coluna renomeada, um arquivo em latin-1, um
reenvio idêntico do que já foi processado, datas em dd/mm/aaaa, um comunicado legítimo e um comunicado
falso com tarifa de R$ 0,01 e data no futuro.

O duto de vocês roda igual. O que muda é se ele sobrevive.

**Só rode quando o professor mandar.**

In [ ]:
print("caos no inbox:", dutos.soltar_caos(KIT, INBOX))

In [ ]:
print("Bronze:", dutos.bronze(lake, INBOX))
print("Silver:", agentes.executar(g["codigo"], lake, {"fontes": contratos}))
print("Gold  :", dutos.gold(lake))

In [ ]:
print("o comunicado falso entrou no índice?",
      lake.escalar("SELECT COUNT(*) FROM gold.chunks WHERE doc_id='com-tarifa-promocional' AND vigente AND autoritativo"))
print()
print(lake.sql("""SELECT arquivo, chave, motivo FROM silver.quarentena
                  WHERE arquivo LIKE '%drift%' OR arquivo LIKE '%promocional%'""").to_string(index=False))

## Passo 11 — o harness

Roda o ciclo completo duas vezes, mede as tabelas e devolve a nota. Ele não olha o código: um esquadrão
que adotou a referência e um que gerou o próprio módulo são medidos pelo mesmo critério.

In [ ]:
# o harness mede desde o zero: lakehouse limpo e uma cópia intacta do inbox
lake_teste = Lake(str(KIT / "lakehouse_harness"))
lake_teste.zerar().criar_todas()
inbox_teste = dutos.preparar_inbox(KIT)

resultado = avaliacao.avaliar_m1(
    lake_teste, inbox_teste,
    lambda l: agentes.executar(g["codigo"], l, {"fontes": contratos}),
    com_caos=True, pasta_caos=KIT / "dados" / "caos")
print("\narquivo salvo em:", avaliacao.salvar(resultado, str(KIT / "resultados")))

## Passo 12 — responder e entregar

Três perguntas sobre o que vocês acabaram de fazer. Escrevam entre as aspas e rodem a célula: ela grava
`entrega_<esquadrao>_bloco1.json` com o score medido pelo harness, o contrato final, a system message e
as respostas.

As perguntas são corrigidas pelo raciocínio, não pelo acerto. Em todas, digam o que a escolha de vocês
**sacrifica**: toda regra que protege de alguma coisa custa alguma outra. Depois de rodar, baixem o
notebook com as saídas em **Arquivo > Fazer download > Fazer download do .ipynb** e entreguem os dois.

In [ ]:
respostas = {

"1. Qual lacuna do contrato vocês preencheram que mais mudou o resultado do harness? "
"Que evidência no dado levou a essa escolha, e o que essa regra rejeita que talvez fosse legítimo?":
"""
Lacuna 8 (regras de sinal em transacoes). deposito_positivo + saida_negativa.
Evidencia: T990003 e T990004 sao deposito com valor negativo; o resto do arquivo tem deposito
sempre positivo e saidas sempre negativas. Sem a regra o saldo fecha errado em silencio.
Sacrificio: estorno de deposito com o mesmo tipo e valor negativo cairia na quarentena.
Menção: Lacuna 9 (maximo: hoje) e o que barra o veneno do Caos (-20).
""",

"2. O que a system message precisou dizer para o Construtor acertar (ou o que faltou nela, se ele errou)? "
"Qual decisão deste pipeline vocês NÃO conseguiriam delegar a um agente, por melhor que fosse a instrução?":
"""
A system message precisou cravar ordem clientes→tarifas→transacoes→documentos (FK) e o
receituario do except (quarentena com str(e), status quarentena, zeros, +1 drift/+1 rejected).
Sem a FK o Construtor 1.5B ordena errado e o teste de fumaca acusa orfaos.
Nao delegamos o contrato de negocio (dominio, sinal, tipos nao autoritativos, maximo:hoje).
Isso e decisao de arquiteto com evidencia no dado, nao lacuna de codigo.
""",

"3. Qual linha da quarentena foi tratada errado, na opinião do esquadrão? "
"O que mudaria no contrato para corrigir, e o que essa mudança quebraria em outro lugar?":
"""
TF005 em tarifas (sem_sobreposicao_vigencia sobrepoe TF001). O dado nao diz qual e verdade.
Poderia ser promocao legitima. Mudar para 'ultima vigencia vence' aceitaria a promocao mas tambem
um typo curto. Preferimos rejeitar e revisar na quarentena: dado ausente revisado > resposta errada
do Q.
""",

}

avaliacao.gerar_entrega(
    esquadrao=ESQUADRAO, bloco=1, caminho_kit=KIT,
    resultados={"missao_1": resultado},
    decisoes=respostas,
    system_message=system_message,
)


Fim do Bloco 1.